# 带行业偏离约束与单票上限的组合优化（cvxpy 版）

本 Notebook 给出一个可直接运行的 Python 示例，演示如何使用 `cvxpy` 在 **行业偏离 ± 1%**、**单票上限 1%** 以及 **全投资约束** 下，最大化 alpha 预测得分的组合权重。

- 所有代码注释均为中文，便于理解每一步在做什么。
- 图表的标题、图例与坐标轴标签使用英文，避免中文字体环境依赖。
- 生成的 PNG 图片会保存在 `outputs/images` 目录中，便于在 Jupyter Notebook 中展示并在版本库中留存。


In [ ]:
# %% 初始化常用库（所有注释均为中文）
import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 设置绘图风格，并确保输出路径存在
sns.set(style='whitegrid', palette='muted', font_scale=1.1)
OUTPUT_DIR = Path('outputs/images')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



## 组合优化函数：行业偏离约束 + 单票上限

下面的函数基于题目描述实现：
- 目标：最大化 `alpha_scores^T w`。
- 约束：权重和为 1、非负、单票不超过 1%、行业权重相对等权基准偏离不超过 1%。
- 为了便于扩展，`industry_dev` 与 `max_weight` 作为参数暴露。


In [ ]:
def optimize_portfolio_with_industry_constraints(alpha_scores, industry_matrix, industry_dev=0.01, max_weight=0.01, solver=cp.SCS):
    '''基于行业哑变量的线性规划求解最优权重。'''
    # --- 1. 输入转为 numpy，并做基本维度检查（全部注释为中文） ---
    alpha_scores = np.asarray(alpha_scores, dtype=float)
    industry_matrix = np.asarray(industry_matrix, dtype=float)

    num_stocks = alpha_scores.shape[0]
    if industry_matrix.shape[0] != num_stocks:
        raise ValueError('industry_matrix 的行数必须与 alpha_scores 长度一致')

    num_industries = industry_matrix.shape[1]

    # --- 2. 构造等权行业基准向量 b：行业权重 = 行业股票数 / 总股票数 ---
    stock_count_per_industry = industry_matrix.sum(axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        industry_benchmark = stock_count_per_industry / float(num_stocks)
        industry_benchmark = np.nan_to_num(industry_benchmark, nan=0.0)

    # --- 3. 定义决策变量 w，长度为 N ---
    w = cp.Variable(num_stocks)

    # --- 4. 目标函数：最大化 alpha_scores^T w ---
    objective = cp.Maximize(alpha_scores @ w)

    # --- 5. 构造约束列表 ---
    constraints = []

    # 5.1 权重和为 1（全投资约束）
    constraints.append(cp.sum(w) == 1.0)

    # 5.2 非负约束与单票上限
    constraints.append(w >= 0.0)
    constraints.append(w <= max_weight)

    # 5.3 行业偏离约束：b_k - industry_dev <= (I^T w)_k <= b_k + industry_dev
    industry_weights = industry_matrix.T @ w  # 形状为 (K,)
    for k in range(num_industries):
        if stock_count_per_industry[k] > 0:
            b_k = industry_benchmark[k]
            constraints.append(industry_weights[k] >= b_k - industry_dev)
            constraints.append(industry_weights[k] <= b_k + industry_dev)

    # --- 6. 求解线性规划 ---
    problem = cp.Problem(objective, constraints)
    problem.solve(solver=solver)

    if w.value is None:
        raise RuntimeError('优化问题未找到可行解，可能是约束过于严格')

    # --- 7. 返回最优权重（扁平化为一维向量） ---
    return np.array(w.value).flatten()



## 构造示例数据并运行优化

- 随机生成 50 只股票的 alpha 得分；
- 随机分配到 4 个行业；
- 运行优化器并展示权重结果。


In [ ]:
# %% 随机生成示例数据（设置随机种子保证复现）
np.random.seed(42)
num_stocks = 50
num_industries = 4

# 生成 alpha 得分，假设均值略为正以模拟有一定 alpha 预期
alpha_scores = np.random.normal(loc=0.02, scale=0.05, size=num_stocks)

# 随机行业分配：为每只股票随机选择一个行业并做成 one-hot 矩阵
industry_labels = np.random.choice(num_industries, size=num_stocks)
industry_matrix = np.zeros((num_stocks, num_industries))
industry_matrix[np.arange(num_stocks), industry_labels] = 1

# 运行优化函数
optimal_weights = optimize_portfolio_with_industry_constraints(alpha_scores, industry_matrix)

print('最优权重向量前 10 项:', np.round(optimal_weights[:10], 4))
print('权重和:', optimal_weights.sum())
print('是否满足单票上限（<=1%）:', np.all(optimal_weights <= 0.01 + 1e-9))



## 结果分析：行业权重与基准偏离

计算优化后组合在各行业的权重，与等权基准行业权重进行对比。


In [ ]:
# %% 计算行业权重并与基准对比
industry_weights = industry_matrix.T @ optimal_weights
stock_count_per_industry = industry_matrix.sum(axis=0)
benchmark_industry = stock_count_per_industry / float(num_stocks)

industry_df = pd.DataFrame({
    'Industry': [f'Industry {i+1}' for i in range(num_industries)],
    'Optimized': industry_weights,
    'Benchmark': benchmark_industry,
})

print(industry_df)



## 可视化：权重分布与行业对比（PNG 输出）

下方两幅图：
1. 单只股票权重柱状图；
2. 行业权重与等权基准的对比柱状图。

图表标题与图例使用英文，避免中文字体依赖；图片以 PNG 格式保存到 `outputs/images`。


In [ ]:
# %% 绘制单只股票权重柱状图（英文标题与标签）
fig, ax = plt.subplots(figsize=(12, 4))
stock_indices = np.arange(num_stocks)
ax.bar(stock_indices, optimal_weights, color='#4C72B0')
ax.set_title('Portfolio Weights (Cvxpy Solution)')
ax.set_xlabel('Stock Index')
ax.set_ylabel('Weight')
ax.axhline(0.01, color='red', linestyle='--', linewidth=1.2, label='Max Single Weight')
ax.legend()
plt.tight_layout()
weights_path = OUTPUT_DIR / 'portfolio_weights_cvxpy.png'
plt.savefig(weights_path, dpi=200)
plt.show()
print(f'权重柱状图已保存至: {weights_path}')



In [ ]:
# %% 绘制行业权重对比图（英文标题与标签）
fig, ax = plt.subplots(figsize=(8, 5))
bar_width = 0.35
positions = np.arange(num_industries)
ax.bar(positions - bar_width/2, industry_df['Optimized'], width=bar_width, label='Optimized', color='#55A868')
ax.bar(positions + bar_width/2, industry_df['Benchmark'], width=bar_width, label='Benchmark', color='#C44E52')
ax.set_xticks(positions)
ax.set_xticklabels(industry_df['Industry'])
ax.set_title('Industry Weights vs Benchmark')
ax.set_ylabel('Weight')
ax.legend()
plt.tight_layout()
industry_path = OUTPUT_DIR / 'industry_weights_comparison.png'
plt.savefig(industry_path, dpi=200)
plt.show()
print(f'行业权重对比图已保存至: {industry_path}')



## 小结与进一步扩展

- 当前示例展示了在 **行业偏离 ± 1%** 与 **单票 1% 上限** 下的线性规划求解。
- 若需要加入风险项（如协方差矩阵）或跟踪误差等，可以将目标替换为二次规划，并在 `cvxpy` 中增加相应约束。
- `industry_dev` 与 `max_weight` 参数可在函数调用时调整，以适配不同的风险控制需求。
